<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     align="right"
     width="300"/>

# **Simple GAN on CelebA**
**Architecture:** DCGAN (Radford et al., 2015)
**Dataset:** CelebA (torchvision) · **Framework:** PyTorch · **Platform:** Google Colab

- Aissa Berenice González Fosado
- Daniela de la Torre Gallo
- Clara Paola Aguilar Casillas


This is the **baseline GAN** before AttGAN. Comparing both shows the advantage of
conditional attribute-guided generation.

| | Simple GAN (this notebook) | AttGAN |
|---|---|---|
| Conditioning | None — random noise only | Attribute vector {-1, +1} |
| Control over output | None | Edit 13 specific facial attributes |
| Loss | BCE adversarial only | Adv + Classification + Reconstruction |
| Output resolution | 64x64 | 128x128 |


## *Clone repo & install*

In [ ]:
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git 2>/dev/null || echo 'Repo already cloned'
%cd GAN_Project_DL
!pip install -q -r requirements.txt
print('Setup complete')

## *Verify GPU & config*

In [ ]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change Runtime Type > T4 GPU'
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from config import Config

cfg = Config()
cfg.EXPERIMENT_NAME = 'simple_gan'
cfg.__init__()   # creates results/simple_gan/ and checkpoints/simple_gan/

device = torch.device('cuda')

# Reduce for a quick smoke test:
# cfg.N_EPOCHS = 5

print(f'Experiment  : {cfg.EXPERIMENT_NAME}')
print(f'Epochs      : {cfg.N_EPOCHS}')
print(f'Batch size  : {cfg.BATCH_SIZE}')
print(f'Results dir : {cfg.RESULTS_DIR}')


## *Load CelebA*
torchvision downloads CelebA automatically on first run (~1.4 GB).  
Images are loaded at 128×128 and downsampled to **64×64** inside the training loop.



In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}  (loaded at 128x128, downsampled to 64x64 in training)')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]')

## *Build models*
```
noise z (100 x 1 x 1)
  |-> Generator  (5x ConvTranspose2d + BatchNorm + ReLU -> Tanh)
      fake image (3 x 64 x 64)
        |-> Discriminator  (5x Conv2d + BatchNorm + LeakyReLU -> Sigmoid)
            real/fake scalar

Loss G : BCE( D(G(z)) , 1 )            <- fool discriminator
Loss D : BCE( D(x)    , 1 )            <- real images -> 1
       + BCE( D(G(z)) , 0 )            <- fake images -> 0
```

In [ ]:
from src.simple_gan import build_simple_models

LATENT_DIM = 100
gen, dis = build_simple_models(latent_dim=LATENT_DIM, device=device)


## Train
Sample grids are saved to `results/simple_gan/` every 5 epochs.  
A healthy run: both G and D losses decrease then oscillate — **neither should reach 0**.

In [ ]:
from src.simple_gan import train_simple_gan

g_losses, d_losses = train_simple_gan(
    gen, dis, train_loader, cfg, device, latent_dim=LATENT_DIM
)

---
## Cell 6 — Loss curves

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

## *FID & DACID*
**FID** — Frechet Inception Distance. Standard GAN quality metric. Lower = better.  
**DACID** — custom metric: L2 distance between mean Inception feature vectors. Lower = better.


In [ ]:
# cfg.COMPUTE_METRICS = False  # uncomment to skip

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics_simple_gan
    scores = compute_metrics_simple_gan(gen, test_loader, LATENT_DIM, cfg, device)
    print(f"FID   : {scores['fid']}")
    print(f"DACID : {scores['dacid']}")
else:
    scores = {}
    print('Metrics skipped.')

## *Save checkpoint & metrics.json*

In [ ]:
import json, torch

# Save model weights
ckpt = cfg.CHECKPOINT_DIR / 'simple_gan_final.pt'
torch.save({'gen': gen.state_dict(), 'dis': dis.state_dict()}, ckpt)
print(f'Checkpoint -> {ckpt}')

# Save metrics.json so export_results.py can include this model
payload = {
    'experiment': cfg.EXPERIMENT_NAME,
    'model':      'SimpleGAN',
    'fid':        scores.get('fid'),
    'dacid':      scores.get('dacid'),
    'g_losses':   g_losses,
    'd_losses':   d_losses,
}
out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Metrics  -> {out}')

## *Download results ZIP*

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('simple_gan_results', 'zip', cfg.RESULTS_DIR)
files.download('simple_gan_results.zip')

## *Fallback* 
If the CelebA quota exceeded

Use one of these if Cell 3 raises `FileURLRetrievalError: Too many users`.


In [ ]:
# 1. Go to kaggle.com > Account > Create New Token  (downloads kaggle.json)
# 2. Upload kaggle.json via the Colab Files panel (left sidebar)
# 3. Uncomment and run:

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d jessicali9530/celeba-dataset -p data/
# !unzip -q data/celeba-dataset.zip -d data/

# Then re-run Cell 3
print('Uncomment above lines after uploading kaggle.json')

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.makedirs('data', exist_ok=True)
# !ln -sfn '/content/drive/MyDrive/YOUR_CELEBA_FOLDER' data/celeba
# # Then re-run Cell 3
print('Uncomment above and replace YOUR_CELEBA_FOLDER with your path')